# NN12 TM4 정정 — 못 돈 23시 조각만 다시: TM4(N5의 짝 계획기, 이웃 시간표 + 좌회전 4배)

처음 실행에서 이 정책의 23시 조각 10판이 기록용 대조 코드의 오류로 멈췄습니다. 끝난 220판은 그대로 쓰고, **멈춘 10판만** 고친 코드로 돌립니다. 판단·입력·평가 코드는 바뀌지 않았습니다.
자세한 내용은 `results/explore/nn12_20261103/사전등록.md`의 정정 기록에 있습니다.

정정 노트북은 둘(N5·TM4)과 판정 하나입니다. **둘을 런타임 하나씩 따로** 씁니다. 순서는 상관없고 동시에 돌려도 됩니다. Rn·M4는 다시 돌리지 않습니다.

1. 런타임 유형 **CPU + 높은 RAM(CPU 8개)** 을 고르고 **런타임 → 모두 실행**을 누르세요. GPU는 필요 없습니다. Drive 접근을 허용하세요.
2. 순서: 묶음 검증 → 12개 지도 자료 → 패키지 → 시험 → 끝난 판 하나가 소수점까지 같은지 점검(약 3분) → 본 실행(10판, 약 5분, 어림).
3. 끊기면 다시 **모두 실행**하세요. 끝난 판은 건너뜁니다.
4. 둘이 끝나면 `NN12_judge_fix1`을 실행하고 "끝났어"라고 알려 주세요. 처음 노트북(`NN12_N5`, `NN12_TM4`, `NN12_judge`)은 쓰지 마세요.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
POLICY = "TM4"
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_NN12_fix1.zip"
NET12_ZIP = DRIVE / "FutureTraffic_network12.zip"
MODELS = None                                       # 신경망 모델(읽기만, 해시 대조)
OUT = DRIVE / "explore_nn12" / POLICY
WORKSPACE = Path("/content/FutureTraffic_NN12_fix1")
MIN_CPUS = 8
for path in (ARCHIVE, NET12_ZIP, OUT / "runs", *([MODELS / "seed_42" / "model.pt"] if MODELS else [])):
    assert path.exists(), f"필요한 파일이 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 정책:", POLICY, "| 결과 폴더:", OUT)
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "NN12_V1", "NN12 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("nn12_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON0 설정 {package_manifest['platoon0_config_sha256'][:12]} | 측정 계획 {package_manifest['plan_sha256'][:12]}")

In [ ]:
# 12개 지도 자료(기존 묶음 그대로). 코드는 들어 있지 않고 수요·지도만 있습니다.
with zipfile.ZipFile(NET12_ZIP) as z:
    z.extractall(WORKSPACE)
m12 = json.loads((WORKSPACE / "network12_manifest.json").read_text())
bad = [p for p, h in m12["files"].items() if hashlib.sha256((WORKSPACE / p).read_bytes()).hexdigest() != h]
assert not bad, f"12개 지도 자료 해시 불일치: {bad[:3]}"
assert m12["network12_fingerprint"] == "711875f6c939", "12개 지도 지문이 다릅니다"
print(f"12개 지도 자료: 파일 {len(m12['files'])}개 | 배경 {m12['background']:.0%} | 유턴 {m12['uturn']}")
os.chdir(WORKSPACE)

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_nn12.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 실행 전 점검(약 3분): 이미 끝난 판 하나를 고친 코드로 다시 돌려 저장된 결과와 소수점까지 같은지 봅니다. 결과 값은 보여 주지 않습니다.
CHECK = OUT / "check_fix1"
previous = colab.read_json(CHECK / f"recheck_{POLICY}.json")
if previous and previous.get("ok"):
    print("이전 점검이 통과해 건너뜁니다.")
else:
    args = ["src/explore_nn12.py", "recheck", "--policy", POLICY, "--out", str(OUT), "--work", "/content/n12c"]
    if MODELS:
        args += ["--models", str(MODELS)]
    colab.run_live(args, WORKSPACE, check=False)
r = colab.read_json(CHECK / f"recheck_{POLICY}.json")
assert r and r.get("ok"), f"점검 실패 — 본 실행을 하지 않습니다. 이 칸의 출력을 저에게 보내 주세요: {r and r.get('same')}"
print("점검 통과:", r["same"], "| 판 초", r["seconds"])

In [ ]:
# 2) 본 실행(남은 판만). 끝난 220판은 건너뛰고 23시 조각 10판을 돌립니다. 백그라운드로 돌아 셀을 멈춰도 계속 돕니다.
#    런타임이 끊기면 처음 다섯 칸 → 이 칸을 다시 실행하세요.
assert "colab" in globals(), "처음 다섯 칸(1~5)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
LOGS = OUT / "logs_fix1"

def describe(name):
    done = len(list((OUT / "runs").glob("*/metrics.json")))
    still = [p for p in (OUT / "runs").glob("*.failed.json") if not (OUT / "runs" / p.name.replace(".failed.json", "") / "metrics.json").exists()]
    return f"끝난 판 {done}/230, 아직 못 끝낸 실패 {len(still)}"

args = ["src/explore_nn12.py", "run", "--policy", POLICY, "--out", str(OUT), "--work", "/content/n12", "--workers", str(CPUS)]
if MODELS:
    args += ["--models", str(MODELS)]
pids = colab.launch({f"nn12_{POLICY}": args}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 60, describe)
assert code == 0, "본 실행에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."
print("본 실행 끝.", describe(POLICY), "| N5·TM4가 모두 끝나면 NN12_judge_fix1을 실행하세요.")